**DATA INGESTION PIPELINE**

- Data loading from llamaindex simple directory reader
- Data normaliation for text cleaning
- Data chunking and embedding using llmaindex ingestion pipeline
- Create Milvus database schema and load entities to vector database with indexing

In [ ]:
#import libraries

import re
import unicodedata

from dotenv import load_dotenv
from llama_index.core import Document, SimpleDirectoryReader
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.readers.file import PyMuPDFReader
from pymilvus import DataType, MilvusClient

load_dotenv()

In [ ]:
#load pdf files from given path

file_extractor = {".pdf": PyMuPDFReader()}
reader = SimpleDirectoryReader(input_files=["../data/Analysis of the Effectiveness of ARIMA, SARIMA, and SVR.pdf"], file_extractor=file_extractor)
document = reader.load_data()

In [ ]:
# data normalization/ cleaning

class DataNormalizer:
    """Normalizes text of Documents produced by SimpleDirectoryReader."""

    def __init__(self, documents: list[Document]):
        self.documents = documents

    def normalize(self) -> list[Document]:
        for doc in self.documents:
            doc.set_content(self._normalize_text(doc.text))
        return self.documents

    def _normalize_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text)
        text = self._fix_hyphenation(text)
        text = self._collapse_newlines(text)
        text = self._collapse_whitespace(text)
        return text.strip()

    def _fix_hyphenation(self, text: str) -> str:
        return re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    def _collapse_newlines(self, text: str) -> str:
        text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
        text = re.sub(r"\n{2,}", "\n\n", text)
        return text

    def _collapse_whitespace(self, text: str) -> str:
        return re.sub(r"[ \t]{2,}", " ", text)

In [ ]:
#create normalized data

data_normalizer = DataNormalizer(documents=document)
normalized_text = data_normalizer.normalize()

In [ ]:
# create the pipeline with transformations
# chunking, titleextraction and embedding

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=150, chunk_overlap=0),
        TitleExtractor(),
        OpenAIEmbedding(),
    ]
)

# run the pipeline
nodes = []
for doc in range(len(normalized_text)):

    text = normalized_text[doc]
    node = pipeline.run(documents=[text])
    nodes.extend(node)


  0%|          | 0/1 [00:00<?, ?it/s]2026-08-28 16:59:06,656 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-28 16:59:09,015 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-28 16:59:10,653 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-28 16:59:11,984 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-28 16:59:12,701 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-28 16:59:14,339 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:09<00:00,  9.09s/it]
2026-08-28 16:59:15,467 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
  0%|          | 0/1 [00:00<?, ?it/s]2026-08-28 16:59:15,538 - INFO - Retrying request to /chat/completions in 0.477927 seconds
2026

In [12]:
print("The context length of embeddings:", len(nodes[3].embedding))

The context length of embeddings: 1536


In [ ]:
# create database client

client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus",
)

In [32]:
collection_name = "paper_chunks"


schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=False,
)


schema.add_field(
    field_name="id",
    datatype=DataType.VARCHAR,
    max_length=100,
    is_primary=True,
)

schema.add_field(
    field_name="title",
    datatype=DataType.VARCHAR,
    max_length=500,
)

schema.add_field(
    field_name="page",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="chunk_index",
    datatype=DataType.INT64,
)


schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=10000,
)

schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=1536,
)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'name': 'page', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'chunk_index', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 10000}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [ ]:
# define indexing parameters

index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)

In [ ]:
# create collection with indexing strategy

client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
)

In [ ]:
# load entities to database

chunk_id=0
for node in nodes:

        data = [{
                "id": node.id_,

                "title": node.metadata["document_title"],

                "page": int(node.metadata["source"]),
                
                "chunk_index": chunk_id,

                "text": node.text,

                "embedding": node.embedding,
        }]
        result = client.insert(
                collection_name="paper_chunks",
                data=data
        )
        chunk_id = chunk_id+1
        print(result)

{'insert_count': 1, 'ids': ['656af604-151a-4693-9862-4568cdeba1c8']}
{'insert_count': 1, 'ids': ['f0539c02-81f1-4b80-82ff-f31e0579aed2']}
{'insert_count': 1, 'ids': ['dee980f6-e1a1-4b4a-9c32-12b46d1279c5']}
{'insert_count': 1, 'ids': ['b25c1187-0505-4574-bc77-29f90985ceb7']}
{'insert_count': 1, 'ids': ['2ca81f23-d3b3-4ea7-ace0-20af31fedb0e']}
{'insert_count': 1, 'ids': ['a8e68db4-3526-47fd-a872-10ae07c28364']}
{'insert_count': 1, 'ids': ['67a52de0-8e51-4963-8138-b533402c6171']}
{'insert_count': 1, 'ids': ['446850d5-ad7e-4057-9598-99009b872134']}
{'insert_count': 1, 'ids': ['3362c49f-bd11-41ab-861d-24e41597f974']}
{'insert_count': 1, 'ids': ['65be59e0-14ff-4707-833d-427812cc147a']}
{'insert_count': 1, 'ids': ['2a7c19e2-c895-4af4-96c4-577dcc10a05c']}
{'insert_count': 1, 'ids': ['6ecc018e-81aa-4c5c-8368-8952b0a89e08']}
{'insert_count': 1, 'ids': ['a890b084-c5b6-412e-82cb-945a9bdb753a']}
{'insert_count': 1, 'ids': ['f5b75608-e977-41d3-883a-b3ec6d439724']}
{'insert_count': 1, 'ids': ['44c94

In [56]:
print("Exists:", client.has_collection(collection_name))

count = client.query(
    collection_name=collection_name,
    filter="",
    output_fields=["count(*)"],
)

print("Entity count:", count)

Exists: True
Entity count: data: ["{'count(*)': 130}"], extra_info: {}


In [58]:
results = client.query(
    collection_name=collection_name,
    filter="",
    output_fields=[
        "id",
        "title",
        "page",
        "chunk_index",
        "page",
        "text",
        "embedding",
    ],
    limit=3,
)

for row in results:
    print(row)
    print("-" * 80)

{'id': '00cacaf9-034e-45a6-8c72-59d17ed7ea3f', 'title': 'Advancements in Renewable Energy Technologies and Energy Storage Systems: A Comprehensive Review', 'page': 17, 'chunk_index': 115, 'text': 'Manowska, A.; Rybak, A.; Dylong, A.; Pielot, J. Forecasting of Natural Gas Consumption in Poland Based on ARIMA-LSTM Hybrid Model. Energies 2021, 14, 8597. [CrossRef] 14. Nokeri, T.C. Forecasting Using ARIMA, SARIMA, and the Additive Model. In Implementing Machine Learning for Finance; Apress: Berkeley, CA, USA, 2021.', 'embedding': [0.0027342133689671755, -0.009556743316352367, -0.0065468549728393555, -0.006744508631527424, -0.002160323318094015, -0.0026717963628470898, -0.0049933637492358685, -0.017338069155812263, -0.024980690330266953, -0.028157023712992668, 0.012316964566707611, 0.02582678757607937, -0.020930517464876175, 0.005034975241869688, -0.016921956092119217, -0.002888522343710065, 0.030653705820441246, -0.0027272782754153013, -0.008911767043173313, -0.016602935269474983, -0.00066